# 📊 Analyse des résultats du Baccalauréat par département (2023 - 2025)

Exploration statistique et visuelle des taux de réussite au baccalauréat en France selon les voies de formation, les académies et le genre.

## 1. Importation des bibliothèques et chargement des données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from skimpy import skim
import pygwalker as pyg

# Chargement du jeu de données
df = pd.read_csv('fr-en-baccalaureat-par-departement.csv', sep=';')
df.head()

## 2. Pré-analyse et qualité des données

In [ ]:
# Synthèse statistique complète, types des colonnes et valeurs manquantes
skim(df)

In [ ]:
# Vérification des lignes en doublon
print(f"Nombre de doublons : {df.duplicated().sum()}")

## 3. Exploration interactive avec PyGWalker

Interface visuelle interactive permettant d'explorer librement les variables. Les configurations de graphiques sont enregistrées dans `chart_meta.json`.

In [ ]:
walker = pyg.walk(df, spec="./chart_meta.json")

## 4. Détection des valeurs atypiques (Outliers)

Utilisation de la méthode de l'écart interquartile (IQR) avec un seuil de 2 pour identifier les taux de réussite atypiques.

In [ ]:
def detect_outliers(dataframe, feature):
    Q1 = dataframe[feature].quantile(0.25)
    Q3 = dataframe[feature].quantile(0.75)
    IQR = Q3 - Q1
    return (dataframe[feature] < (Q1 - 2 * IQR)) | (dataframe[feature] > (Q3 + 2 * IQR))

df[detect_outliers(df, "Taux de réussite à l'examen")]

### 🔍 Zoom sur les Outliers du Taux de Réussite
La méthode IQR identifie 5 observations comme des valeurs atypiques inférieures (taux de réussite entre 62 % et 67 %).

Après vérification des variables qualitatives (`Voie` et `Libellé département`), il ne s'agit pas d'erreurs de saisie mais de données réelles : ce sont des résultats de la voie professionnelle dans des territoires ultramarins (Guyane et Mayotte). Ces entités cumulent les taux de réussite historiquement plus bas de la voie professionnelle et des défis structurels spécifiques. Elles sont donc conservées dans l'analyse.

## 5. Agrégations par voie, académie et département

In [ ]:
# Taux de réussite moyen par voie de formation
df.groupby('Voie')["Taux de réussite à l'examen"].mean().round(2)

In [ ]:
# Taux de réussite moyen par académie (ordre croissant)
df.groupby("Libellé académie")["Taux de réussite à l'examen"].mean().round(1).sort_values(ascending=True)

In [ ]:
# Effectif moyen de présents à l'examen par académie
df.groupby("Libellé académie")["Nombre de présents à l'examen"].mean().round(1).sort_values(ascending=True)

In [ ]:
# Export d'un fichier agrégé par département pour utilisation dans Power BI
df_departement = (
    df.groupby(['Code département', 'Libellé département'])["Taux de réussite à l'examen"]
    .mean()
    .round(1)
    .reset_index()
)
df_departement.to_csv('reussite_bac_par_departement.csv', index=False, sep=';')

## 6. Visualisations des distributions et disparités

In [ ]:
# Distribution globale du taux de réussite
plt.figure(figsize=(10, 6))
plt.hist(df["Taux de réussite à l'examen"], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution globale du taux de réussite au baccalauréat', fontsize=12, fontweight='bold')
plt.xlabel('Taux de réussite (%)')
plt.ylabel('Fréquence')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Distribution du taux de réussite par voie de formation
plt.figure(figsize=(10, 6))

for voie in df['Voie'].unique():
    subset = df[df['Voie'] == voie]
    plt.hist(subset["Taux de réussite à l'examen"], bins=15, alpha=0.6, label=voie, edgecolor='black')

plt.title("Distribution du taux de réussite par voie de formation", fontsize=12, fontweight='bold')
plt.xlabel("Taux de réussite (%)")
plt.ylabel("Nombre d'observations")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('distribution_voies.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Boîtes à moustaches (Boxplot) par voie de formation
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Voie', y="Taux de réussite à l'examen", palette="Set2", hue='Voie')

plt.title("Comparaison des distributions du taux de réussite par voie", fontsize=12, fontweight='bold')
plt.xlabel("Voie de formation")
plt.ylabel("Taux de réussite (%)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('boxplot_voies.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Vérification des meilleurs taux de réussite en voie professionnelle (> 96 %)
df[(df['Voie'] == 'Professionnelle') & (df["Taux de réussite à l'examen"] > 96)]

## 7. Analyse par genre et validation statistique (Test de Student)

In [ ]:
# Comparaison globale par genre
print("--- Taux de réussite moyen par Genre ---")
print(df.groupby('Genre')["Taux de réussite à l'examen"].mean().round(1))

# Analyse croisée : Voie et Genre
print("\n--- Taux de réussite croisé (Voie et Genre) ---")
analyse_croisee = df.groupby(['Voie', 'Genre'])["Taux de réussite à l'examen"].mean().round(1).reset_index()
print(analyse_croisee)

In [ ]:
# Graphique à barres comparatif Voie / Genre
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df,
    x='Voie',
    y="Taux de réussite à l'examen",
    hue='Genre',
    palette='muted',
    edgecolor='black'
)

plt.title("Comparaison des taux de réussite au baccalauréat par Voie et Genre", fontsize=13, fontweight='bold')
plt.xlabel("Voie de formation", fontsize=11)
plt.ylabel("Taux de réussite moyen (%)", fontsize=11)
plt.ylim(70, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('comparaison_genre_voie.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Test de Student (Welch) pour évaluer la significativité de l'écart en voie professionnelle
voie_pro_f = df[(df["Voie"] == "Professionnelle") & (df["Genre"] == "Féminin")]["Taux de réussite à l'examen"]
voie_pro_m = df[(df["Voie"] == "Professionnelle") & (df["Genre"] == "Masculin")]["Taux de réussite à l'examen"]

t_stat, p_value = stats.ttest_ind(voie_pro_f, voie_pro_m, equal_var=False)

print("--- TEST STATISTIQUE (SciPy) ---")
print(f"T-statistic : {t_stat:.4f}")
print(f"P-value : {p_value:.4e}")
if p_value < 0.05:
    print("👉 L'écart de réussite entre les genres en voie professionnelle est STATISTIQUEMENT SIGNIFICATIF (p < 0.05).")
else:
    print("👉 L'écart n'est pas statistiquement significatif au seuil de 5%.")